# 🌲 EvaluaPlus - Entrenamiento con ResNet50
Clasifica imágenes de madera como **buena** o **defectuosa**.

**Antes de empezar:**
1. Ve a `Entorno de ejecución → Cambiar tipo → T4 GPU`
2. Pon tu API key de Roboflow
3. Ejecuta todas las celdas en orden

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

In [ ]:
!pip install roboflow -q

## 3. Descargar dataset de Roboflow

In [ ]:
API_KEY = 'PON_TU_API_KEY_AQUI'  # <- Cambia esto

from roboflow import Roboflow
from pathlib import Path
import shutil

rf = Roboflow(api_key=API_KEY)
project = rf.workspace('faron-ace-fs10i').project('wood-defect-instant')
versions = project.versions()
version_num = versions[0].version if versions else 1
print(f'Descargando versión {version_num}...')
project.version(version_num).download('yolov8', location='dataset_raw')
print('✅ Dataset descargado')

## 4. Preparar dataset para clasificación

In [ ]:
import shutil
from pathlib import Path

# Estructura: dataset/good/ y dataset/defect/
for clase in ['train/good', 'train/defect', 'val/good', 'val/defect']:
    Path(f'dataset/{clase}').mkdir(parents=True, exist_ok=True)

raw = Path('dataset_raw')
extensiones = {'.jpg', '.jpeg', '.png'}

def tiene_defecto(img_path):
    label = Path(str(img_path).replace('images', 'labels')).with_suffix('.txt')
    return label.exists() and label.stat().st_size > 0

# Recopilar imágenes
train_imgs = []
for ext in extensiones:
    train_imgs.extend((raw / 'train' / 'images').rglob(f'*{ext}'))

val_imgs = []
for split in ['valid', 'test']:
    p = raw / split / 'images'
    if p.exists():
        for ext in extensiones:
            val_imgs.extend(p.rglob(f'*{ext}'))

# Copiar a carpetas de clase
for i, img in enumerate(train_imgs):
    clase = 'defect' if tiene_defecto(img) else 'good'
    shutil.copy(img, f'dataset/train/{clase}/{i:05d}{img.suffix}')

for i, img in enumerate(val_imgs):
    clase = 'defect' if tiene_defecto(img) else 'good'
    shutil.copy(img, f'dataset/val/{clase}/{i:05d}{img.suffix}')

# Contar
for split in ['train', 'val']:
    for clase in ['good', 'defect']:
        n = len(list(Path(f'dataset/{split}/{clase}').glob('*')))
        print(f'{split}/{clase}: {n} imágenes')

## 5. Entrenar ResNet50

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

# Transformaciones
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Datasets
train_dataset = datasets.ImageFolder('dataset/train', transform=transform_train)
val_dataset = datasets.ImageFolder('dataset/val', transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f'Clases: {train_dataset.classes}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

# Modelo ResNet50 preentrenado
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)  # 2 clases: good, defect
model = model.to(device)

# Entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

EPOCHS = 10
mejor_val_acc = 0

for epoch in range(EPOCHS):
    # Entrenamiento
    model.train()
    train_loss, train_correct = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # Validación
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = train_correct / len(train_dataset) * 100
    val_acc = val_correct / len(val_dataset) * 100
    print(f'Epoch {epoch+1}/{EPOCHS} - Train acc: {train_acc:.1f}% - Val acc: {val_acc:.1f}%')

    if val_acc > mejor_val_acc:
        mejor_val_acc = val_acc
        torch.save(model.state_dict(), 'mejor_modelo.pth')
        print(f'  ✅ Mejor modelo guardado ({val_acc:.1f}%)')

    scheduler.step()

print(f'\n🎉 Entrenamiento completado. Mejor val acc: {mejor_val_acc:.1f}%')

## 6. Guardar modelo con metadata

In [ ]:
import torch
from torchvision import models
import torch.nn as nn

# Cargar mejor modelo
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load('mejor_modelo.pth', map_location='cpu'))
model.eval()

# Guardar con metadata
torch.save({
    'model_state_dict': model.state_dict(),
    'clases': ['defect', 'good'],  # orden de ImageFolder (alfabético)
    'arquitectura': 'resnet50',
    'input_size': (224, 224),
}, 'evalua_plus_model.pth')

print('✅ Modelo guardado como evalua_plus_model.pth')

## 7. Descargar modelo

In [ ]:
from google.colab import files
files.download('evalua_plus_model.pth')
print('✅ Descargando modelo...')